In [5]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/movie_data.csv', encoding='utf-8')

X_train, X_test, y_train, y_test = train_test_split(
    df['review'],
    df['sentiment'],
    test_size=0.3,
    random_state=7
)

# X_train = X_train[:20000,]
# y_train = y_train[:20000,]

tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=100000, min_df=5, dtype=np.float32)

X_train = tfidf.fit_transform(X_train)
X_test = tfidf.transform(X_test)



In [6]:
from torch.utils.data import DataLoader, TensorDataset
import torch
from lib import *

# dataset = JointSparseDataset(X_train, y_train)
# data_loader = DataLoader(dataset, 64, shuffle=True, collate_fn= sparse_collate)

dataset = JointDataset(X_train, y_train)
data_loader = DataLoader(dataset, 64, shuffle=True)

In [7]:
from lib import NeuralNet

input_shape = X_train.shape[1]

model = NeuralNet(input_shape)
optimizer = torch.optim.Adam(model.parameters(), lr = 0.01, weight_decay= 0)
train_model(model, optimizer, data_loader)



Device found: cuda
Train time = 46.00018104400078 sec


NeuralNet(
  (fc1): Linear(in_features=100000, out_features=256, bias=True)
  (fc2): Linear(in_features=256, out_features=256, bias=True)
  (fc3): Linear(in_features=256, out_features=256, bias=True)
  (fc4): Linear(in_features=256, out_features=64, bias=True)
  (output): Linear(in_features=64, out_features=1, bias=True)
)

In [10]:
model.eval()

test_loader = DataLoader(JointDataset(X_test, y_test), batch_size=64, shuffle=False)
evaluate_model(model, test_loader)


      

Test Accuracy: 0.8957333333333334
